# Module 1 - Programming Assignment

## General Directions

1. You must follow the Programming Requirements outlined on Canvas.
2. The Notebook should be cleanly and fully executed before submission.
3. You should change the name of this file to be your JHED id. For example, `jsmith299.ipynb` although Canvas will change it to something else... :/

<div style="background: lemonchiffon; margin:20px; padding: 20px;">
    <strong>Important</strong>
    <p>
        You should always read the entire assignment before beginning your work, so that you know in advance what the requested output will be and can plan your implementation accordingly.
    </p>
</div>

# State Space Search with A* Search

You are going to implement the A\* Search algorithm for navigation problems.


## Motivation

Search is often used for path-finding in video games. Although the characters in a video game often move in continuous spaces,
it is trivial to layout a "waypoint" system as a kind of navigation grid over the continuous space. Then if the character needs
to get from Point A to Point B, it does a line of sight (LOS) scan to find the nearest waypoint (let's call it Waypoint A) and
finds the nearest, LOS waypoint to Point B (let's call it Waypoint B). The agent then does a A* search for Waypoint B from Waypoint A to find the shortest path. The entire path is thus Point A to Waypoint A to Waypoint B to Point B.

We're going to simplify the problem by working in a grid world. The symbols that form the grid have a special meaning as they
specify the type of the terrain and the cost to enter a grid cell with that type of terrain:

```
token   terrain    cost 
🌾       plains     1
🌲       forest     3
⛰       hills      5
🐊       swamp      7
🌋       mountains  impassible
```

We can think of the raw format of the map as being something like:

```
🌾🌾🌾🌾🌲🌾🌾
🌾🌾🌾🌲🌲🌲🌾
🌾⛰⛰⛰🌾🌾🌾
🌾🌾⛰⛰🌾🌾🌾
🌾🌾⛰🌾🌾🌲🌲
🌾🌾🌾🌾🌲🌲🌲
🌾🌾🌾🌾🌾🌾🌾
```

## The World

Given a map like the one above, we can easily represent each row as a `List` and the entire map as `List of Lists`:

In [1]:
full_world = [
['🌾', '🌾', '🌾', '🌾', '🌾', '🌲', '🌲', '🌲', '🌲', '🌲', '🌲', '🌲', '🌲', '🌲', '🌲', '🌾', '🌾', '🌾', '🌾', '🌾', '🌾', '🌾', '🌾', '🌾', '🌾', '🌾', '🌾'],
['🌾', '🌾', '🌾', '🌾', '🌾', '🌾', '🌾', '🌲', '🌲', '🌲', '🌲', '🌲', '🌲', '🌲', '🌲', '🌲', '🌾', '🌾', '🌋', '🌋', '🌋', '🌋', '🌋', '🌋', '🌋', '🌾', '🌾'],
['🌾', '🌾', '🌾', '🌾', '🌋', '🌋', '🌲', '🌲', '🌲', '🌲', '🌲', '🌲', '🌲', '🌲', '🌲', '🌲', '🌲', '🌋', '🌋', '🌋', '⛰', '⛰', '⛰', '🌋', '🌋', '⛰', '⛰'],
['🌾', '🌾', '🌾', '🌾', '⛰', '🌋', '🌋', '🌋', '🌲', '🌲', '🌲', '🌲', '🐊', '🐊', '🌲', '🌲', '🌲', '🌲', '🌲', '🌾', '🌾', '⛰', '⛰', '🌋', '🌋', '⛰', '🌾'],
['🌾', '🌾', '🌾', '⛰', '⛰', '🌋', '🌋', '🌲', '🌲', '🌾', '🌾', '🐊', '🐊', '🐊', '🐊', '🌲', '🌲', '🌲', '🌾', '🌾', '🌾', '⛰', '🌋', '🌋', '🌋', '⛰', '🌾'],
['🌾', '⛰', '⛰', '⛰', '🌋', '🌋', '⛰', '⛰', '🌾', '🌾', '🌾', '🌾', '🐊', '🐊', '🐊', '🐊', '🐊', '🌾', '🌾', '🌾', '🌾', '🌾', '⛰', '🌋', '⛰', '🌾', '🌾'],
['🌾', '⛰', '⛰', '🌋', '🌋', '⛰', '⛰', '🌾', '🌾', '🌾', '🌾', '⛰', '🌋', '🌋', '🌋', '🐊', '🐊', '🐊', '🌾', '🌾', '🌾', '🌾', '🌾', '⛰', '🌾', '🌾', '🌾'],
['🌾', '🌾', '⛰', '⛰', '⛰', '⛰', '⛰', '🌾', '🌾', '🌾', '🌾', '🌾', '🌾', '⛰', '🌋', '🌋', '🌋', '🐊', '🐊', '🐊', '🌾', '🌾', '⛰', '⛰', '⛰', '🌾', '🌾'],
['🌾', '🌾', '🌾', '⛰', '⛰', '⛰', '🌾', '🌾', '🌾', '🌾', '🌾', '🌾', '⛰', '⛰', '🌋', '🌋', '🌾', '🐊', '🐊', '🌾', '🌾', '⛰', '⛰', '⛰', '🌾', '🌾', '🌾'],
['🌾', '🌾', '🌾', '🐊', '🐊', '🐊', '🌾', '🌾', '⛰', '⛰', '⛰', '🌋', '🌋', '🌋', '🌋', '🌾', '🌾', '🌾', '🐊', '🌾', '⛰', '⛰', '⛰', '🌾', '🌾', '🌾', '🌾'],
['🌾', '🌾', '🐊', '🐊', '🐊', '🐊', '🐊', '🌾', '⛰', '⛰', '🌋', '🌋', '🌋', '⛰', '🌾', '🌾', '🌾', '🌾', '🌾', '⛰', '🌋', '🌋', '🌋', '⛰', '🌾', '🌾', '🌾'],
['🌾', '🐊', '🐊', '🐊', '🐊', '🐊', '🌾', '🌾', '⛰', '🌋', '🌋', '⛰', '🌾', '🌾', '🌾', '🌾', '🐊', '🐊', '🌾', '🌾', '⛰', '🌋', '🌋', '⛰', '🌾', '🌾', '🌾'],
['🐊', '🐊', '🐊', '🐊', '🐊', '🌾', '🌾', '⛰', '⛰', '🌋', '🌋', '⛰', '🌾', '🐊', '🐊', '🐊', '🐊', '🌾', '🌾', '🌾', '⛰', '🌋', '⛰', '🌾', '🌾', '🌾', '🌾'],
['🌾', '🐊', '🐊', '🐊', '🐊', '🌾', '🌾', '⛰', '🌲', '🌲', '⛰', '🌾', '🌾', '🌾', '🌾', '🐊', '🐊', '🐊', '🐊', '🌾', '🌾', '⛰', '🌾', '🌾', '🌾', '🌾', '🌾'],
['🌾', '🌾', '🌾', '🌾', '🌋', '🌾', '🌾', '🌲', '🌲', '🌲', '🌲', '⛰', '⛰', '⛰', '⛰', '🌾', '🐊', '🐊', '🐊', '🌾', '🌾', '⛰', '🌋', '⛰', '🌾', '🌾', '🌾'],
['🌾', '🌾', '🌾', '🌋', '🌋', '🌋', '🌲', '🌲', '🌲', '🌲', '🌲', '🌲', '🌋', '🌋', '🌋', '⛰', '⛰', '🌾', '🐊', '🌾', '⛰', '🌋', '🌋', '⛰', '🌾', '🌾', '🌾'],
['🌾', '🌾', '🌋', '🌋', '🌲', '🌲', '🌲', '🌲', '🌲', '🌲', '🌲', '🌲', '🌲', '🌲', '🌋', '🌋', '🌋', '🌾', '🌾', '🌋', '🌋', '🌋', '🌾', '🌾', '🌾', '🌾', '🌾'],
['🌾', '🌾', '🌾', '🌋', '🌋', '🌲', '🌲', '🌲', '🌲', '🌲', '🌲', '🌲', '🌲', '🌲', '🌲', '🌲', '🌋', '🌋', '🌋', '🌋', '🌾', '🌾', '🌾', '🌾', '🌾', '🌾', '🌾'],
['🌾', '🌾', '🌾', '🌋', '🌋', '🌋', '🌲', '🌲', '🌲', '🌲', '🌲', '🌲', '🌲', '🌲', '🌾', '🌾', '🌾', '⛰', '⛰', '🌾', '🌾', '🌾', '🌾', '🌾', '🌾', '🌾', '🌾'],
['🌾', '🌾', '🌾', '🌾', '🌋', '🌋', '🌋', '🌲', '🌲', '🌲', '🌲', '🌲', '🌲', '🌾', '🌾', '🌾', '🌾', '🌾', '🌾', '🌾', '🌾', '🌾', '🌾', '🐊', '🐊', '🐊', '🐊'],
['🌾', '🌾', '⛰', '⛰', '⛰', '⛰', '🌋', '🌋', '🌲', '🌲', '🌲', '🌲', '🌲', '🌾', '🌋', '🌾', '🌾', '🌾', '🌾', '🌾', '🐊', '🐊', '🐊', '🐊', '🐊', '🐊', '🐊'],
['🌾', '🌾', '🌾', '🌾', '⛰', '⛰', '⛰', '🌋', '🌋', '🌋', '🌲', '🌲', '🌋', '🌋', '🌾', '🌾', '🌾', '🌾', '🌾', '🌾', '🐊', '🐊', '🐊', '🐊', '🐊', '🐊', '🐊'],
['🌾', '🌾', '🌾', '🌾', '🌾', '🌾', '⛰', '⛰', '⛰', '🌋', '🌋', '🌋', '🌋', '🌾', '🌾', '🌾', '🌾', '⛰', '⛰', '🌾', '🌾', '🐊', '🐊', '🐊', '🐊', '🐊', '🐊'],
['🌾', '⛰', '⛰', '🌾', '🌾', '⛰', '⛰', '⛰', '⛰', '⛰', '🌾', '🌾', '🌾', '🌾', '🌾', '⛰', '⛰', '🌋', '🌋', '⛰', '⛰', '🌾', '🐊', '🐊', '🐊', '🐊', '🐊'],
['⛰', '🌋', '⛰', '⛰', '⛰', '⛰', '🌾', '🌾', '🌾', '🌾', '🌾', '🌋', '🌋', '🌋', '⛰', '⛰', '🌋', '🌋', '🌾', '🌋', '🌋', '⛰', '⛰', '🐊', '🐊', '🐊', '🐊'],
['⛰', '🌋', '🌋', '🌋', '⛰', '🌾', '🌾', '🌾', '🌾', '🌾', '⛰', '⛰', '🌋', '🌋', '🌋', '🌋', '⛰', '⛰', '⛰', '⛰', '🌋', '🌋', '🌋', '🐊', '🐊', '🐊', '🐊'],
['⛰', '⛰', '🌾', '🌾', '🌾', '🌾', '🌾', '🌾', '🌾', '🌾', '🌾', '🌾', '⛰', '⛰', '⛰', '⛰', '⛰', '🌾', '🌾', '🌾', '🌾', '⛰', '⛰', '⛰', '🌾', '🌾', '🌾']
]

In [2]:
for line in full_world:
    print("".join(line))

🌾🌾🌾🌾🌾🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌾🌾🌾🌾🌾🌾🌾🌾🌾🌾🌾🌾
🌾🌾🌾🌾🌾🌾🌾🌲🌲🌲🌲🌲🌲🌲🌲🌲🌾🌾🌋🌋🌋🌋🌋🌋🌋🌾🌾
🌾🌾🌾🌾🌋🌋🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌋🌋🌋⛰⛰⛰🌋🌋⛰⛰
🌾🌾🌾🌾⛰🌋🌋🌋🌲🌲🌲🌲🐊🐊🌲🌲🌲🌲🌲🌾🌾⛰⛰🌋🌋⛰🌾
🌾🌾🌾⛰⛰🌋🌋🌲🌲🌾🌾🐊🐊🐊🐊🌲🌲🌲🌾🌾🌾⛰🌋🌋🌋⛰🌾
🌾⛰⛰⛰🌋🌋⛰⛰🌾🌾🌾🌾🐊🐊🐊🐊🐊🌾🌾🌾🌾🌾⛰🌋⛰🌾🌾
🌾⛰⛰🌋🌋⛰⛰🌾🌾🌾🌾⛰🌋🌋🌋🐊🐊🐊🌾🌾🌾🌾🌾⛰🌾🌾🌾
🌾🌾⛰⛰⛰⛰⛰🌾🌾🌾🌾🌾🌾⛰🌋🌋🌋🐊🐊🐊🌾🌾⛰⛰⛰🌾🌾
🌾🌾🌾⛰⛰⛰🌾🌾🌾🌾🌾🌾⛰⛰🌋🌋🌾🐊🐊🌾🌾⛰⛰⛰🌾🌾🌾
🌾🌾🌾🐊🐊🐊🌾🌾⛰⛰⛰🌋🌋🌋🌋🌾🌾🌾🐊🌾⛰⛰⛰🌾🌾🌾🌾
🌾🌾🐊🐊🐊🐊🐊🌾⛰⛰🌋🌋🌋⛰🌾🌾🌾🌾🌾⛰🌋🌋🌋⛰🌾🌾🌾
🌾🐊🐊🐊🐊🐊🌾🌾⛰🌋🌋⛰🌾🌾🌾🌾🐊🐊🌾🌾⛰🌋🌋⛰🌾🌾🌾
🐊🐊🐊🐊🐊🌾🌾⛰⛰🌋🌋⛰🌾🐊🐊🐊🐊🌾🌾🌾⛰🌋⛰🌾🌾🌾🌾
🌾🐊🐊🐊🐊🌾🌾⛰🌲🌲⛰🌾🌾🌾🌾🐊🐊🐊🐊🌾🌾⛰🌾🌾🌾🌾🌾
🌾🌾🌾🌾🌋🌾🌾🌲🌲🌲🌲⛰⛰⛰⛰🌾🐊🐊🐊🌾🌾⛰🌋⛰🌾🌾🌾
🌾🌾🌾🌋🌋🌋🌲🌲🌲🌲🌲🌲🌋🌋🌋⛰⛰🌾🐊🌾⛰🌋🌋⛰🌾🌾🌾
🌾🌾🌋🌋🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌋🌋🌋🌾🌾🌋🌋🌋🌾🌾🌾🌾🌾
🌾🌾🌾🌋🌋🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌋🌋🌋🌋🌾🌾🌾🌾🌾🌾🌾
🌾🌾🌾🌋🌋🌋🌲🌲🌲🌲🌲🌲🌲🌲🌾🌾🌾⛰⛰🌾🌾🌾🌾🌾🌾🌾🌾
🌾🌾🌾🌾🌋🌋🌋🌲🌲🌲🌲🌲🌲🌾🌾🌾🌾🌾🌾🌾🌾🌾🌾🐊🐊🐊🐊
🌾🌾⛰⛰⛰⛰🌋🌋🌲🌲🌲🌲🌲🌾🌋🌾🌾🌾🌾🌾🐊🐊🐊🐊🐊🐊🐊
🌾🌾🌾🌾⛰⛰⛰🌋🌋🌋🌲🌲🌋🌋🌾🌾🌾🌾🌾🌾🐊🐊🐊🐊🐊🐊🐊
🌾🌾🌾🌾🌾🌾⛰⛰⛰🌋🌋🌋🌋🌾🌾🌾🌾⛰⛰🌾🌾🐊🐊🐊🐊🐊🐊
🌾⛰⛰🌾🌾⛰⛰⛰⛰⛰🌾🌾🌾🌾🌾⛰⛰🌋🌋⛰⛰🌾🐊🐊🐊🐊🐊
⛰🌋⛰⛰⛰⛰🌾🌾🌾🌾🌾🌋🌋🌋⛰⛰🌋🌋🌾🌋🌋⛰⛰🐊🐊🐊🐊
⛰🌋🌋🌋⛰🌾🌾🌾🌾🌾⛰⛰🌋🌋🌋🌋⛰⛰⛰⛰🌋🌋🌋🐊🐊🐊🐊
⛰⛰🌾🌾🌾🌾🌾🌾🌾🌾🌾🌾⛰⛰⛰⛰⛰🌾🌾🌾🌾⛰⛰⛰🌾🌾🌾


## Warning

One implication of this representation is that (x, y) is world[ y][ x] so that (3, 2) is world[ 2][ 3] and world[ 7][ 9] is (9, 7). Yes, there are many ways to do this. I picked this representation because when you look at it, it *looks* like a regular x, y cartesian grid and it's easy to print out.

It is often easier to begin your programming by operating on test input that has an obvious solution. If we had a small 7x7 world with the following characteristics:

In [3]:
small_world = [
    ['🌾', '🌲', '🌲', '🌲', '🌲', '🌲', '🌲'],
    ['🌾', '🌲', '🌲', '🌲', '🌲', '🌲', '🌲'],
    ['🌾', '🌲', '🌲', '🌲', '🌲', '🌲', '🌲'],
    ['🌾', '🌾', '🌾', '🌾', '🌾', '🌾', '🌾'],
    ['🌲', '🌲', '🌲', '🌲', '🌲', '🌲', '🌾'],
    ['🌲', '🌲', '🌲', '🌲', '🌲', '🌲', '🌾'],
    ['🌲', '🌲', '🌲', '🌲', '🌲', '🌲', '🌾']
]

**what do you expect the policy would be?** Think about it for a bit. This will help you with your programming and debugging.

## States and State Representation

The canonical pieces of a State Space Search problem are the States, Actions, Transitions and Costs. 

We'll start with the state representation. For the navigation problem, a state is the current position of the agent, `(x,y)`. The entire set of possible states is implicitly represented by the world map.

## Actions and Transitions

Next we need to specify the actions. In general, there are a number of different possible action sets in such a world. The agent might be constrained to move north/south/east/west or diagonal moves might be permitted as well (or really anything). When combined with the set of States, the *permissible* actions forms the Transition set.

Rather than enumerate the Transition set directly, for this problem it's easier to calculate the available actions and transitions on the fly. This can be done by specifying a *movement model* as offsets to the current state and then checking to see which of the potential successor states are actually permitted. This can be done in the successor function mentioned in the pseudocode.

One such example of a movement model is shown below.

In [4]:
MOVES = [(0,-1), (1,0), (0,1), (-1,0)]

## Costs

We can encode the costs described above in a `Dict`:

In [5]:
COSTS = { '🌾': 1, '🌲': 3, '⛰': 5, '🐊': 7}

## Specification

You will implement a function called `a_star_search` that takes the parameters and returns the value as specified below. The return value is going to look like this:

`[(0,1), (0,1), (0,1), (1,0), (1,0), (1,0), (1,0), (1,0), (1,0), (0,1), (0,1), (0,1)]`

You should also implement a function called `pretty_print_path`. 
The `pretty_print_path` function prints an ASCII representation of the path generated by the `a_star_search` on top of the terrain map. 
For example, for the test world, it would print this:

```
⏬🌲🌲🌲🌲🌲🌲
⏬🌲🌲🌲🌲🌲🌲
⏬🌲🌲🌲🌲🌲🌲
⏩⏩⏩⏩⏩⏩⏬
🌲🌲🌲🌲🌲🌲⏬
🌲🌲🌲🌲🌲🌲⏬
🌲🌲🌲🌲🌲🌲🎁
```

using ⏩,⏪,⏫ ⏬ to represent actions and `🎁` to represent the goal. (Note the format of the output...there are no spaces, commas, or extraneous characters). You are printing the path over the terrain.
This is an impure function (because it has side effects, the printing, before returning anything).

Note that in Python:
```
> a = ["*", "-", "*"]
> "".join(a)
*-*
```
* Do not print raw data structures
* Do not insert unneeded/unrequested spaces
* Use the provided emojis!

### Additional comments

As Python is an interpreted language, you're going to need to insert all of your functions *before* the actual `a_star_search` function implementation. 
Do not make unwarranted assumptions (for example, do not assume that the start is always (0, 0).
Do not refer to global variables, pass them as parameters (functional programming).

Simple and correct is better than inefficient and incorrect, or worse, incomplete.
For example, you can use a simple List, with some helper functions, as a Stack or a Queue or a Priority Queue.
Avoid the Python implementations of HeapQ, PriorityQueue implementation unless you are very sure about what you're doing as they require *immutable* keys.

In [6]:
from typing import List, Tuple, Dict, Callable
from copy import deepcopy

*add as many markdown and code cells here as you need for helper functions. We have added `heuristic` for you*

<a id="heuristic"></a>
## heuristic

A heuristic function approximates the cost to reach the end goal from a given state. In A* search, the model sums the known cost of to reach a given point and the heuristic estimate cost to reach the goal as the priority key when exploring the states. The Manhattan distance is the sum of the minimum number of steps required horizontally and vertically to reach goal state. As the problem statement dictates that one can only move vertically (up and down), and horizontally (left and right), the Manhattan distance between a given position and the end goal position is a suitable option to estimate the cost to reach the end goal.  As the lowest cost of the state in the problem is 1, we can estimate that the each step in the heuristic function is of cost 1. 

* **position** Tuple[int, int]: the starting location of the bot, `(x, y)`.
* **goal** Tuple[int, int]: the desired goal position for the bot, `(x, y)`.

**returns** int: the estimated cost to reach the end goal as calculated by the Manhattan distance.


In [7]:
def heuristic(position: Tuple[int, int], goal: Tuple[int, int]): 
    vertical_steps = abs(goal[0] - position[0])
    horizontal_steps = abs(goal[1] - position[1])
    total_steps = vertical_steps + horizontal_steps  # Manhattan distance
    return total_steps

In [8]:
#each test case consists of a starting point, goal point and expected result
tests = [
    [(0, 0), (0, 0), 0],
    [(4, 2), (0, 0), 6],
    [(-9, 2), (7, 3), 17],
    [(-3, -3), (-2, -2), 2],
    [(1, 3), (1, 3), 0],
    [(-4, 5), (2, -1), 12],
    ]

for test in tests: 
    assert heuristic(test[0], test[1]) == test[2]

<a id="push_frontier"></a>
## push_frontier

In A* search, the frontier utilizes a priority queue to prioritize states of the least cost, f(n). The state of the lowest f(n) is popped from the priority queue to be explored. This function adds a state to the frontier and sorts the frontier by the priority key, f(n), in ascending order.

* **frontier** List[Tuple[int, Tuple[int, int]]]: the queue holding the cost f(n) and the position
* **position** Tuple[int, int]: the position to be pushed onto the priority queue
* **f** int: the calculated f(n) value to act as the priority key
  
**returns** updated **frontier** List[Tuple[int, Tuple[int, int]]]

In [9]:
from operator import itemgetter

def push_frontier(frontier: List[Tuple[int, Tuple[int, int]]], position: Tuple[int, int], f: int): 
    frontier.append((f, position))
    frontier = sorted(frontier, key = itemgetter(0))
    return frontier

In [10]:
#each test case consists of a starting point, goal point and expected result
target = [(0, (0, 0)), (1, (3, 1)), (2, (9, 1))]
test_frontier = [(0, (0, 0)), (2, (9, 1))]  
sorted_frontier = push_frontier(test_frontier, (3, 1), 1)
assert sorted_frontier == target

target = [(-4, (1, 2)), (-2, (4, 2)), (9, (2, 0))]
test_frontier = [(9, (2, 0)), (-2, (4, 2))]  
sorted_frontier = push_frontier(test_frontier, (1, 2), -4)
assert sorted_frontier == target

target = [(0, (3, 2)), (0, (0, 0)), (3, (2, 1))]
test_frontier = [(3, (2, 1)), (0, (3, 2))]  
sorted_frontier = push_frontier(test_frontier, (0, 0), 0)
assert sorted_frontier == target

<a id="successor"></a>
## successor

* **world** List[List[str]]: the actual context for the navigation problem.
* **current_position** Tuple[int, int]: the current location of the bot, `(x, y)`.
* **goal** Tuple[int, int]: the desired goal position for the bot, `(x, y)`.
* **costs** Dict[str, int]: is a `Dict` of costs for each type of terrain in **world**.
* **moves** List[Tuple[int, int]]: the legal movement model expressed in offsets in **world**.
* **heuristic** Callable: is a heuristic function, $h(n)$.
* **g_n** dict[Tuple[int, int], int]: is a 'Dict' of the calculated g(n) for each position
* **h_n** dict[Tuple[int, int], int]: is a 'Dict' of the calculated h(n) for each position
* **f_n** dict[Tuple[int, int], int]: is a 'Dict' of the calculated f(n) for each position
* **parents** dict[Tuple[int, int], [Tuple[int, int]]]: is a 'Dict' of the parent states of each position, if any found
* **frontier** List[Tuple[int, Tuple[int, int]]]: the queue holding the cost f(n) and the position
  
**returns** updated **g_n** dict[Tuple[int, int], int], updated **h_n** dict[Tuple[int, int], int], updated **f_n** dict[Tuple[int, int], int], updated **parents** dict[Tuple[int, int], Optional[Tuple[int, int]]], updated **frontier** List[Tuple[int, Tuple[int, int]]]

In [11]:
def successor(world: List[List[str]], current_position: Tuple[int, int], goal: Tuple[int, int], costs: Dict[str, int], moves: List[Tuple[int, int]], heuristic: Callable, g_n: Dict[Tuple[int, int], int], h_n: Dict[Tuple[int, int], int], f_n: Dict[Tuple[int, int], int], parents: Dict[Tuple[int, int], [Tuple[int, int]]], frontier: List[Tuple[int, Tuple[int, int]]]): 
    for y, x in moves: 
        successor = current_position[0] + y, current_position[1] + x
        if 0 <= successor[0] < len(world) and 0 <= successor[1] < len(world[0]):
            if world[successor[0]][successor[1]] in costs:
                g = g_n[current_position] + costs[world[successor[0]][successor[1]]]

                if successor not in g_n or g < g_n[successor]: 
                    g_n[successor] = g
                    h_n[successor] = heuristic(successor, goal)
                    f_n[successor] = g_n[successor] + h_n[successor]
                    parents[successor] = current_position
                    frontier = push_frontier(frontier, successor, f_n[successor])

    return g_n, h_n, f_n, parents, frontier
            



In [12]:
mini_world = [
    ['🌾', '🌲', '🌲', '🌲', '🌲'],
    ['🌾', '🌲', '🌲', '🌲', '🌲'],
    ['🌾', '🌲', '🌲', '🌲', '🌲'],
    ['🌾', '🌾', '🌾', '🌾', '🌾'],
    ['🌲', '🌲', '🌲', '🌲', '🌾'],
    ['🌲', '🌲', '🌲', '🌲', '🌾'],
    ['🌲', '🌲', '🌲', '🌲', '🌾']
    ]
start_test = (0,0)
goal_test = (1,1)
g_test = {start_test: 0}
h_test = {start_test: heuristic(start_test, goal_test)}
f_test = {start_test: g_test[start_test] + h_test[start_test]}
parents_test = {start_test: None}
frontier_test = []

g_test, h_test, f_test, parents_test, frontier_test =successor(mini_world, start_test, goal_test, COSTS, MOVES, heuristic, g_test, h_test, f_test, parents_test, frontier_test)

assert len(frontier_test) == 2
assert parents_test[(1,0)] == (0,0)
assert parents_test[(0,1)] == (0,0)
assert len(g_test) == 3
assert len(h_test) == 3
assert len(f_test) == 3

<a id="draw_path"></a>
## draw_path

Once the goal state is found via A* search, the pathway from the start position to the goal post must be retraced. A dictionary consisting of the parent states of each state explored is used to trace back to the start position, resulting in the optimum pathway found by the A* search method. 

* **current_position** Tuple[int, int]: the current location of the bot, `(x, y)`.
* **parents** dict[Tuple[int, int], Optional[Tuple[int, int]]]: is a 'Dict' of the parent states of each position, if any found
  
**returns** path List[Tuple[int, int]]

In [13]:
def draw_path(current_position: Tuple[int, int], parents: dict[Tuple[int, int], [Tuple[int, int]]]): 
    pathway = []

    while current_position is not None: 
        pathway.append(current_position)
        current_position = parents[current_position]

    pathway.reverse()
    return pathway

In [14]:
parents_test = {(0,0):(1,0), (1,0):(2,0), (2,0):(3,0), (3,0):(4,0), (4,0):(5,0), (5,0):None}

assert len((draw_path((0,0), parents_test))) == 6

assert draw_path((3,0), parents_test) == [(5, 0), (4, 0), (3, 0)]

assert draw_path((5,0), parents_test) == [(5, 0)]


<a id="a_star_search"></a>
## a_star_search

*add documentation/description of the algorithm here; connect it to the theory!*

* **world** List[List[str]]: the actual context for the navigation problem.
* **start** Tuple[int, int]: the starting location of the bot, `(x, y)`.
* **goal** Tuple[int, int]: the desired goal position for the bot, `(x, y)`.
* **costs** Dict[str, int]: is a `Dict` of costs for each type of terrain in **world**.
* **moves** List[Tuple[int, int]]: the legal movement model expressed in offsets in **world**.
* **heuristic** Callable: is a heuristic function, $h(n)$.


**returns** List[Tuple[int, int]]: the offsets needed to get from start state to the goal as a `List`.


In [15]:
def a_star_search( world: List[List[str]], start: Tuple[int, int], goal: Tuple[int, int], costs: Dict[str, int], moves: List[Tuple[int, int]], heuristic: Callable) -> List[Tuple[int, int]]:
    g_n = {start: 0}
    h_n = {start: heuristic(start, goal)}
    f_n = {start: g_n[start] + h_n[start]}
    parents = {start: None}
    frontier = []
    
    frontier = push_frontier(frontier, start, 0)

    while frontier: 
        current_position = frontier[0][1]
        frontier.pop(0)
        if current_position == goal: 
            pathway = draw_path(current_position, parents)
            return pathway
        g_n, h_n, f_n, parents, frontier = successor(world, current_position, goal, costs, moves, heuristic, g_n, h_n, f_n, parents, frontier)

    return None # if no path was found

In [16]:
mini_world = [
    ['🌾', '🌲', '🌲', '🌲', '🌲'],
    ['🌾', '🌲', '🌲', '🌲', '🌲'],
    ['🌾', '🌲', '🌲', '🌲', '🌲'],
    ['🌾', '🌾', '🌾', '🌾', '🌾'],
    ['🌲', '🌲', '🌲', '🌲', '🌾'],
    ['🌲', '🌲', '🌲', '🌲', '🌾'],
    ['🌲', '🌲', '🌲', '🌲', '🌾']
    ]

start_test = (3, 0)
goal_test = (4, 4)

path = a_star_search(mini_world, start_test, goal_test, COSTS, MOVES, heuristic)
assert len(path) == 6
assert path[0] == (3, 0)
assert path[-1] == (4, 4)

## Actions

We can encode the actions used to navigate the world (up, down, left, right) as symbols in a `Dict`. Each symbol is associated with the vertical change, and horizontal change (successor state - current state) as a tuple. 

In [17]:
ACTIONS = {(0, 1): '⏩', (1, 0): '⏬', (0, -1): '⏪', (-1, 0): '⏫'}

## pretty_print_path

This function provides a visualization representation of the path generated by the A* algorithm in context of the terrain. The arrows represent the action taken at each state. The gift character represent the goal. 

* **world** List[List[str]]: the world (terrain map) for the path to be printed upon.
* **path** List[Tuple[int, int]]: the path from start to goal, in offsets.
* **start** Tuple[int, int]: the starting location for the path.
* **goal** Tuple[int, int]: the goal location for the path.
* **costs** Dict[str, int]: the costs for each action.

**returns** int - The path cost.

In [18]:
def pretty_print_path( world: List[List[str]], path: List[Tuple[int, int]], start: Tuple[int, int], goal: Tuple[int, int], COSTS: Dict[str, int], actions: Dict[Tuple[int, int], str]):
    new_map = deepcopy(world)
    total_cost = 0
    
    new_map[goal[0]][goal[1]] = '🎁'

    for i in range(0, len(path)-1):   #skip last tile in pathway / goal state
        total_cost = total_cost + COSTS[world[path[i][0]][path[i][1]]]
        
        vertical_change = path[i+1][0] - path[i][0]
        horizontal_change = path[i+1][1] - path[i][1]
        new_map[path[i][0]][path[i][1]] = ACTIONS[(vertical_change, horizontal_change)]
        
    for i in new_map: 
        print("".join(i))
        
    return total_cost

In [19]:
mini_world = [
    ['🌾', '🌲', '🌲', '🌲', '🌲'],
    ['🌾', '🌲', '🌲', '🌲', '🌲'],
    ['🌾', '🌲', '🌲', '🌲', '🌲'],
    ['🌾', '🌾', '🌾', '🌾', '🌾'],
    ['🌲', '🌲', '🌲', '🌲', '🌾'],
    ['🌲', '🌲', '🌲', '🌲', '🌾'],
    ['🌲', '🌲', '🌲', '🌲', '🌾']
    ]

path1_test = [(3, 0), (3, 1), (3, 2), (3, 3), (3, 4), (4, 4)]
path2_test = [(0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (4, 1)]
path3_test = [(5, 1), (5, 2), (5, 3)]
print("Test 1")
assert pretty_print_path(mini_world, path1_test, (3,0), (4,4), COSTS, ACTIONS) == 5
print("\nTest 2")
assert pretty_print_path(mini_world, path2_test, (0,0), (4,1), COSTS, ACTIONS) == 7
print("\nTest 3")
assert pretty_print_path(mini_world, path3_test, (5,0), (5,3), COSTS, ACTIONS) == 6

Test 1
🌾🌲🌲🌲🌲
🌾🌲🌲🌲🌲
🌾🌲🌲🌲🌲
⏩⏩⏩⏩⏬
🌲🌲🌲🌲🎁
🌲🌲🌲🌲🌾
🌲🌲🌲🌲🌾

Test 2
⏬🌲🌲🌲🌲
⏬🌲🌲🌲🌲
⏬🌲🌲🌲🌲
⏬🌾🌾🌾🌾
⏩🎁🌲🌲🌾
🌲🌲🌲🌲🌾
🌲🌲🌲🌲🌾

Test 3
🌾🌲🌲🌲🌲
🌾🌲🌲🌲🌲
🌾🌲🌲🌲🌲
🌾🌾🌾🌾🌾
🌲🌲🌲🌲🌾
🌲⏩⏩🎁🌾
🌲🌲🌲🌲🌾


## Problems

## Problem 1. 

Execute `a_star_search` and `pretty_print_path` for the `small_world`.

If you change any values while developing your code, make sure you change them back! (Better yet, don't do it. Copy them elsewhere and change the values, then delete those experiments).

In [20]:
small_start = (0, 0)
small_goal = (len(small_world[0]) - 1, len(small_world) - 1)
small_path = a_star_search(small_world, small_start, small_goal, COSTS, MOVES, heuristic)
small_path_cost = pretty_print_path(small_world, small_path, small_start, small_goal, COSTS, ACTIONS)
print(f"total path cost: {small_path_cost}")
print(small_path)

⏬🌲🌲🌲🌲🌲🌲
⏬🌲🌲🌲🌲🌲🌲
⏬🌲🌲🌲🌲🌲🌲
⏩⏩⏩⏩⏩⏩⏬
🌲🌲🌲🌲🌲🌲⏬
🌲🌲🌲🌲🌲🌲⏬
🌲🌲🌲🌲🌲🌲🎁
total path cost: 12
[(0, 0), (1, 0), (2, 0), (3, 0), (3, 1), (3, 2), (3, 3), (3, 4), (3, 5), (3, 6), (4, 6), (5, 6), (6, 6)]


## Problem 2

Execute `a_star_search` and `print_path` for the `full_world`

In [21]:
full_start = (0, 0)
full_goal = (len(full_world[0]) - 1, len(full_world) - 1)
full_path = a_star_search(full_world, full_start, full_goal, COSTS, MOVES, heuristic)
full_path_cost = pretty_print_path(full_world, full_path, full_start, full_goal, COSTS, ACTIONS)
print(f"total path cost: {full_path_cost}")
print(full_path)

⏬🌾🌾🌾🌾🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌾🌾🌾🌾🌾🌾🌾🌾🌾🌾🌾🌾
⏬🌾🌾🌾🌾🌾🌾🌲🌲🌲🌲🌲🌲🌲🌲🌲🌾🌾🌋🌋🌋🌋🌋🌋🌋🌾🌾
⏬🌾🌾🌾🌋🌋🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌋🌋🌋⛰⛰⛰🌋🌋⛰⛰
⏬🌾🌾🌾⛰🌋🌋🌋🌲🌲🌲🌲🐊🐊🌲🌲🌲🌲🌲🌾🌾⛰⛰🌋🌋⛰🌾
⏬🌾🌾⛰⛰🌋🌋🌲🌲🌾🌾🐊🐊🐊🐊🌲🌲🌲🌾🌾🌾⛰🌋🌋🌋⛰🌾
⏬⛰⛰⛰🌋🌋⛰⛰🌾🌾🌾🌾🐊🐊🐊🐊🐊🌾🌾🌾🌾🌾⛰🌋⛰🌾🌾
⏬⛰⛰🌋🌋⛰⛰🌾🌾🌾🌾⛰🌋🌋🌋🐊🐊🐊🌾🌾🌾🌾🌾⛰🌾🌾🌾
⏬🌾⛰⛰⛰⛰⛰🌾🌾🌾🌾🌾🌾⛰🌋🌋🌋🐊🐊🐊🌾🌾⛰⛰⛰🌾🌾
⏬🌾🌾⛰⛰⛰🌾🌾🌾🌾🌾🌾⛰⛰🌋🌋🌾🐊🐊🌾🌾⛰⛰⛰🌾🌾🌾
⏬🌾🌾🐊🐊🐊🌾🌾⛰⛰⛰🌋🌋🌋🌋🌾🌾🌾🐊🌾⛰⛰⛰🌾🌾🌾🌾
⏬🌾🐊🐊🐊🐊🐊🌾⛰⛰🌋🌋🌋⛰🌾🌾🌾🌾🌾⛰🌋🌋🌋⛰🌾🌾🌾
⏬🐊🐊🐊🐊🐊🌾🌾⛰🌋🌋⛰🌾🌾🌾🌾🐊🐊🌾🌾⛰🌋🌋⛰🌾🌾🌾
⏬🐊🐊🐊🐊🌾🌾⛰⛰🌋🌋⛰🌾🐊🐊🐊🐊🌾🌾🌾⛰🌋⛰🌾🌾🌾🌾
⏬🐊🐊🐊🐊🌾🌾⛰🌲🌲⛰🌾🌾🌾🌾🐊🐊🐊🐊🌾🌾⛰🌾🌾🌾🌾🌾
⏬🌾🌾🌾🌋🌾🌾🌲🌲🌲🌲⛰⛰⛰⛰🌾🐊🐊🐊🌾🌾⛰🌋⛰🌾🌾🌾
⏬🌾🌾🌋🌋🌋🌲🌲🌲🌲🌲🌲🌋🌋🌋⛰⛰🌾🐊🌾⛰🌋🌋⛰🌾🌾🌾
⏬🌾🌋🌋🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌋🌋🌋🌾🌾🌋🌋🌋🌾🌾🌾🌾🌾
⏬🌾🌾🌋🌋🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌲🌋🌋🌋🌋🌾🌾🌾🌾🌾🌾🌾
⏬🌾🌾🌋🌋🌋🌲🌲🌲🌲🌲🌲🌲🌲🌾🌾🌾⛰⛰🌾🌾🌾🌾🌾🌾🌾🌾
⏬🌾🌾🌾🌋🌋🌋🌲🌲🌲🌲🌲🌲🌾🌾🌾🌾🌾🌾🌾🌾🌾🌾🐊🐊🐊🐊
⏬🌾⛰⛰⛰⛰🌋🌋🌲🌲🌲🌲🌲🌾🌋🌾🌾🌾🌾🌾🐊🐊🐊🐊🐊🐊🐊
⏬🌾🌾🌾⛰⛰⛰🌋🌋🌋🌲🌲🌋🌋🌾🌾🌾🌾🌾🌾🐊🐊🐊🐊🐊🐊🐊
⏩⏩⏩⏬🌾🌾⛰⛰⛰🌋🌋🌋🌋🌾🌾🌾🌾⛰⛰🌾🌾🐊🐊🐊🐊🐊🐊
🌾⛰⛰⏩⏬⛰⛰⛰⛰⛰🌾🌾🌾🌾🌾⛰⛰🌋🌋⛰⛰🌾🐊🐊🐊🐊🐊
⛰🌋⛰⛰⏬⛰🌾🌾🌾🌾🌾🌋🌋🌋⛰⛰🌋🌋🌾🌋🌋⛰⛰🐊🐊🐊🐊
⛰🌋🌋🌋⏬🌾🌾🌾🌾🌾⛰⛰🌋🌋🌋🌋⛰⛰⛰⛰🌋🌋🌋🐊🐊🐊🐊
⛰⛰🌾🌾⏩⏩⏩⏩⏩⏩⏩⏩⏩⏩⏩⏩⏩⏩⏩⏩⏩⏩⏩⏩⏩⏩🎁
total path cost: 98
[(0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (5, 0), (6, 0), (7, 0), (8, 0), (9, 0), (10, 0), (11, 0), (12, 0), (13, 0), (14, 0), (15, 0), (16, 0), (17, 0), (18, 0), (19, 0), (20, 0), (21, 0), (22, 0), (22, 1), (22, 2), (22, 3),

## Comments

(This is the place to leave me comments)

## To think about for future assignments...

This first assignment may not have been difficult for you if you've encountered A* search before in your Algorithms course. In preparation for future assignments that build on State Space Search, you can think about the following or even do an implementation if you like. You should **not** submit it as part of this assignment.

In several future assignments, we will have a need for a "plain ol'" Depth First Search algorithm.

1. Implement DFS Search to solve the problem presented in this programming assignment. Try to be as general as possible (don't hard code anything you can pass as a formal parameter).
2. Can you implement DFS Search as a higher order function and supply your own `is_goal`, `successors`, and `path` functions? How do you handle *state*?
3. Can you write a version of DFS that returns all the solutions?

In one future assignment a Breadth First Search algorithm will be very handy. Can you implement a search algorithm that changes whether it uses DFS or BFS by parameterization?

## Before You Submit...

1. Did you provide output exactly as requested?
2. Did you follow the Programming Requirements on Canvas?

Do not submit any other files.